# Run the validation on new data/ from configuration files 
## Step 1: Re-initialise the Data Context from Lakehouse Files

In [1]:
import great_expectations as gx

path_to_local_context = '/lakehouse/default/Files'

# initialise the data context from the Lakehouse Files
context = gx.get_context(project_root_dir=path_to_local_context)


StatementMeta(, 982058b2-1816-4346-9cec-d8ec22b086d9, 5, Finished, Available, Finished, False)

In [ ]:
import json
from great_expectations.util import convert_to_json_serializable

# Load cleaned data from bronze layer to be validated
bronze_df_validation = spark.read.parquet("Files/bronze/cleaned_data")

# configure
batch_parameters = {"dataframe": bronze_df_validation}

checkpoint = context.checkpoints.get("gx_checkpoint_1")
file_identifier = "abfss://21a5c954-bf7b-4072-b2e6-389ef4287915@onelake.dfs.fabric.microsoft.com/86585eb0-bf45-4c35-8892-3892ddd9b3bb/Tables/bronze/CustomerChurn".split("/")[-1]
runid = gx.RunIdentifier(run_name=f"Test run - {file_identifier}")

results = checkpoint.run(batch_parameters=batch_parameters, run_id=runid)

json_string = results.describe()
print(json_string)


## Step 2: Handle results
Now we have some results, we want to perform some actions: 
- log the results in a validation log Lakehouse table
- if success, write to Silver Table
- if failure, throw an exception to be handled by data pipeline 

In [ ]:
from datetime import datetime
import pandas as pd

def load_to_silver(validated_bronze): 
    validated_bronze.write.format("delta").mode("overwrite").saveAsTable("silver.CustomerChurn")
    #mssparkutils.notebook.exit()


def parse_and_load_checkpoint_result(results): 

    validation_result = next(iter(results.run_results.values()))
   
    
    # Extract statistiks and metadata direct from objects
    success = results.success
    stats = validation_result.statistics
    meta = validation_result.meta
    

    run_name = results.run_id.run_name if results.run_id else "N/A"
    run_time_str = results.run_id.run_time if results.run_id else datetime.now().isoformat()
    
    # parse time stamp (ISO-Format)
    parsed_time = datetime.strptime(str(run_time_str)[:19], "%Y-%m-%d %H:%M:%S")
    
    restructured = {
        "run_name": [run_name], 
        "run_time": [parsed_time],
        "validation_result": [success], 
        "evaluated_expectations": [stats.get("evaluated_expectations", 0)],
        "successful_expectations": [stats.get("successful_expectations", 0)],
        "unsuccessful_expectations": [stats.get("unsuccessful_expectations", 0)],
        "success_percent": [stats.get("success_percent", 0.0)],
        "expectation_suite_name": [meta.get("expectation_suite_name", "N/A")]
    }

    pandas_df = pd.DataFrame(restructured)
    spark_df = spark.createDataFrame(pandas_df)
    spark_df.write.format("delta").mode("append").save("Tables/validation_results")

    return success

# --- Run PIPELINE ---

# 1. Run Checkpoint (returns GX 1.x CheckpointResult object)
results = checkpoint.run(batch_parameters={"dataframe": bronze_df_validation})

# 2. Call Function with CheckpointResultObject
success = parse_and_load_checkpoint_result(results)

# 3. branch for pipeline to see if validation with Great Expectations was successfull
if success: 
    print("✅ Validation successfull! Load data to silver layer ...")
    load_to_silver(bronze_df_validation)
else: 
    print(f"❌ Validation failure!")
    raise Exception("Handle the exception in the data pipeline.")
